# Regression Models Template – Linear, Lasso, Elastic Net, Ridge

Plantilla para modelado predictivo basado en regresión, siguiendo las etapas del desarrollo de modelos. Este notebook incluye únicamente **Linear Regression**, **Lasso**, **Elastic Net** y **Ridge**.

## Contenido

* [1. Enunciado del problema](#1)
* [2. Inicio – Carga de librerías y datos](#2)
* [3. Análisis exploratorio](#3)
* [4. Preparación de datos](#4)
* [5. Evaluación de algoritmos](#5)
* [6. Afinación de hiperparámetros (Grid Search)](#6)
* [7. Finalización del modelo](#7)

---
# 1. Enunciado del problema <a id='1'></a>

Objetivo de este notebook:

- Trabajar un problema de modelado predictivo de principio a fin.
- Usar transformaciones de datos para mejorar el rendimiento.
- Usar afinación de hiperparámetros (Grid Search) para mejorar el rendimiento.

**Modelos implementados:**

- Linear Regression (OLS)
- Lasso (L1)
- Elastic Net (L1 + L2)
- Ridge (L2)

---
# 2. Inicio – Carga de librerías y datos <a id='2'></a>

## 2.1. Carga de librerías

In [ ]:
# Instalar dependencias (ejecutar una vez si da error)
%pip install openpyxl -q

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from matplotlib import pyplot
from pandas.plotting import scatter_matrix
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV

from sklearn.linear_model import LinearRegression, Lasso, ElasticNet, Ridge

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from pickle import dump, load

## 2.2. Carga de datos

In [ ]:
import subprocess
import sys

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'])

excel_file = 'data_usd-clp.xlsx'
dataset = pd.read_excel(excel_file, engine='openpyxl')

# Configurar columna de fecha como índice
dataset = dataset.set_index(dataset.columns[0])
dataset.index = pd.to_datetime(dataset.index)

print("Tipo de dataset:", type(dataset))
print("Dimensiones:", dataset.shape)

In [ ]:
# Variables: primera columna objetivo (USDCLP), resto predictores
Y = dataset.iloc[:, 0]
X = dataset.iloc[:, 1:]

print("Variable objetivo:", Y.name)
print("Predictores:", list(X.columns))

---
# 3. Análisis exploratorio <a id='3'></a>

## 3.1. Estadísticas descriptivas

In [ ]:
print("Dimensiones:", dataset.shape)
print("\nTipos de datos:")
dataset.dtypes

In [ ]:
pd.set_option('display.width', 100)
pd.set_option('precision', 3)
dataset.head(5)

In [ ]:
dataset.describe()

## 3.2. Visualización de datos

In [ ]:
# Histogramas
dataset.hist(figsize=(12, 10))
pyplot.tight_layout()
pyplot.suptitle('Histogramas de variables', y=1.02)
pyplot.show()

In [ ]:
# Matriz de correlación
corr = dataset.corr()
fig, ax = pyplot.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, ax=ax)
pyplot.title('Matriz de correlación')
pyplot.xticks(rotation=45, ha='right')
pyplot.tight_layout()
pyplot.show()

In [ ]:
# Scatter matrix (muestra de variables para no saturar)
numeric_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()
cols_plot = numeric_cols[:5] if len(numeric_cols) >= 5 else numeric_cols
scatter_matrix(dataset[cols_plot], figsize=(12, 10), alpha=0.5)
pyplot.suptitle('Matriz de dispersión', y=1.02)
pyplot.show()

---
# 4. Preparación de datos <a id='4'></a>

## 4.1. Limpieza de datos

In [ ]:
# Eliminar filas con NaN
dataset_clean = dataset.dropna()
print("Filas originales:", len(dataset))
print("Filas después de limpieza:", len(dataset_clean))

## 4.2. División Train/Test (series temporales)

In [ ]:
# División temporal: 80% entrenamiento, 20% prueba
split_idx = int(len(dataset_clean) * 0.8)

X_train = dataset_clean.iloc[:split_idx, 1:]
Y_train = dataset_clean.iloc[:split_idx, 0]
X_validation = dataset_clean.iloc[split_idx:, 1:]
Y_validation = dataset_clean.iloc[split_idx:, 0]

print("Entrenamiento: X", X_train.shape, "Y", Y_train.shape)
print("Prueba: X", X_validation.shape, "Y", Y_validation.shape)

## 4.3. Transformación de datos (StandardScaler)

In [ ]:
# Estandarización: media 0, desv. estándar 1 (recomendado para Ridge/Lasso/ElasticNet)
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_validation_scaled = scaler.transform(X_validation)

print("Transformación aplicada: StandardScaler")
print("Media train (aprox. 0):", X_train_scaled.mean(axis=0).round(4))
print("Desv. train (aprox. 1):", X_train_scaled.std(axis=0).round(4))

---
# 5. Evaluación de algoritmos <a id='5'></a>

## 5.1. Definición de modelos (Linear, Lasso, Elastic Net, Ridge)

In [ ]:
models = []
models.append(('LR', LinearRegression()))
models.append(('Lasso', Lasso(alpha=0.1)))
models.append(('EN', ElasticNet(alpha=0.1, l1_ratio=0.5)))
models.append(('Ridge', Ridge(alpha=0.1)))

## 5.2. Evaluación con validación cruzada

In [ ]:
results = []
names = []
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models:
    cv_results = cross_val_score(model, X_train_scaled, Y_train, cv=kfold, scoring='neg_mean_squared_error')
    results.append(cv_results)
    names.append(name)
    msg = "%s: %.4f (%.4f)" % (name, cv_results.mean(), cv_results.std())
    print(msg)

In [ ]:
# Comparación visual
fig = pyplot.figure(figsize=(10, 5))
fig.suptitle('Comparación de MSE (Cross-Val, neg MSE)')
ax = fig.add_subplot(111)
pyplot.boxplot(results)
ax.set_xticklabels(names)
pyplot.ylabel('Neg. Mean Squared Error')
pyplot.show()

## 5.3. Métricas en muestra de validación

In [ ]:
# Entrenar cada modelo en train y evaluar en validation
print("Métricas en validación (datos escalados):\n")

for name, model in models:
    model.fit(X_train_scaled, Y_train)
    pred = model.predict(X_validation_scaled)
    mse = mean_squared_error(Y_validation, pred)
    mae = mean_absolute_error(Y_validation, pred)
    r2 = r2_score(Y_validation, pred)
    print("%s:  MSE=%.4f  MAE=%.4f  R2=%.4f" % (name, mse, mae, r2))

---
# 6. Afinación de hiperparámetros (Grid Search) <a id='6'></a>

In [ ]:
# Grid Search para Lasso, Elastic Net y Ridge
param_grid_lasso = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]}
param_grid_en = {'alpha': [0.01, 0.1, 1.0], 'l1_ratio': [0.2, 0.5, 0.8]}
param_grid_ridge = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]}

grids = [
    ('Lasso', Lasso(), param_grid_lasso),
    ('ElasticNet', ElasticNet(), param_grid_en),
    ('Ridge', Ridge(), param_grid_ridge)
]

best_models = {}

for name, model, param_grid in grids:
    grid = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, Y_train)
    best_models[name] = grid.best_estimator_
    print("%s - Mejores params: %s (MSE=%.4f)" % (name, grid.best_params_, -grid.best_score_))

---
# 7. Finalización del modelo <a id='7'></a>

## 7.1. Resultados en test con modelos afinados

In [ ]:
# Linear Regression (sin afinación)
lr = LinearRegression()
lr.fit(X_train_scaled, Y_train)
best_models['LR'] = lr

print("Métricas finales en validación:\n")
for name, model in best_models.items():
    pred = model.predict(X_validation_scaled)
    mse = mean_squared_error(Y_validation, pred)
    r2 = r2_score(Y_validation, pred)
    print("%s:  MSE=%.4f  R2=%.4f" % (name, mse, r2))

## 7.2. Coeficientes (interpretación)

In [ ]:
feature_names = X_train.columns.tolist()

for name, model in best_models.items():
    coef = model.coef_ if hasattr(model, 'coef_') else None
    if coef is not None:
        print("\n%s - Coeficientes:" % name)
        for f, c in zip(feature_names, coef):
            print("  %s: %.4f" % (f[:30], c))

## 7.3. Guardar modelo para uso posterior

In [ ]:
# Ejemplo: guardar el mejor modelo según criterio (ej. Ridge con menor MSE)
scores = {}
for name, model in best_models.items():
    pred = model.predict(X_validation_scaled)
    scores[name] = mean_squared_error(Y_validation, pred)

best_name = min(scores, key=scores.get)
best_model = best_models[best_name]

dump(best_model, open('modelo_regresion_final.pkl', 'wb'))
dump(scaler, open('scaler_final.pkl', 'wb'))

print("Modelo guardado: %s" % best_name)
print("Archivos: modelo_regresion_final.pkl, scaler_final.pkl")